# Feature Engineering — Sistema ELO
**Objetivo:** calcular ratings ELO sobre 49k partidos históricos y analizar su poder predictivo en Mundiales.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.extractor import load_results, filter_world_cups, add_outcome
from src.features import compute_elo_ratings, save_current_elo

## 1. Calcular ELO sobre todos los partidos históricos

In [ ]:
import time

df_all = load_results()
df_all_clean = df_all.dropna(subset=['home_score', 'away_score']).copy()

t0 = time.time()
df_elo, ratings = compute_elo_ratings(df_all_clean)
elapsed = time.time() - t0

print(f"Partidos procesados: {len(df_all_clean):,}")
print(f"Equipos con rating:  {len(ratings)}")
print(f"Tiempo de cómputo:   {elapsed:.2f}s")

# Guardar ELO actual para uso en la app
save_current_elo(ratings)
print("elo_current.json guardado.")

## 2. Distribución global de ELO

In [ ]:
df_ratings = pd.Series(ratings, name='elo').reset_index()
df_ratings.columns = ['team', 'elo']
df_ratings = df_ratings.sort_values('elo', ascending=False).reset_index(drop=True)

print(f"Media global: {df_ratings['elo'].mean():.1f} (debe ser ~1500 por construcción)")
print(f"Std: {df_ratings['elo'].std():.1f}  |  Min: {df_ratings['elo'].min():.1f}  |  Max: {df_ratings['elo'].max():.1f}")

fig = px.histogram(df_ratings, x='elo', nbins=40,
                   title='Distribución de ratings ELO — todos los equipos',
                   color_discrete_sequence=['#C8102E'])
fig.add_vline(x=1500, line_dash='dash', line_color='white', annotation_text='ELO inicial (1500)')
fig.update_layout(xaxis_title='ELO', yaxis_title='Equipos', bargap=0.05)
fig.show()

## 3. Top 25 equipos por ELO actual

In [ ]:
top25 = df_ratings.head(25)

# Colombia en el ranking
col_rank = df_ratings[df_ratings['team'] == 'Colombia'].index[0] + 1
print(f"Colombia — Rank #{col_rank} | ELO: {df_ratings[df_ratings['team']=='Colombia']['elo'].values[0]:.1f}")

colors = ['#FFD700' if t == 'Colombia' else '#C8102E' for t in top25['team']]

fig = go.Figure(go.Bar(
    x=top25['elo'], y=top25['team'],
    orientation='h',
    marker_color=colors,
    text=top25['elo'].round(0).astype(int),
    textposition='outside'
))
fig.update_layout(
    title='Top 25 selecciones por ELO actual (post junio 2026)',
    xaxis_title='ELO', yaxis={'categoryorder': 'total ascending'},
    height=650
)
fig.add_vline(x=1500, line_dash='dot', line_color='gray')
fig.show()

## 4. ELO en partidos de Mundial — poder predictivo

In [ ]:
df_wc = filter_world_cups(df_all).dropna(subset=['home_score', 'away_score']).copy()
df_wc = add_outcome(df_wc)
df_wc['year'] = df_wc['date'].dt.year

wc_elo = df_wc.merge(
    df_elo[['date', 'home_team', 'away_team', 'elo_home', 'elo_away', 'elo_diff']],
    on=['date', 'home_team', 'away_team'], how='left'
)

print("ELO_DIFF promedio por resultado:")
print(wc_elo.groupby('outcome')['elo_diff'].agg(['mean', 'std', 'count']).round(2).to_string())

label_map = {'home_win': 1, 'draw': 0, 'away_win': -1}
wc_elo['outcome_num'] = wc_elo['outcome'].map(label_map)
corr = wc_elo[['elo_diff', 'outcome_num']].corr().iloc[0, 1]
print(f"\nCorrelación elo_diff vs outcome (numérico): {corr:.3f}")

In [ ]:
# Boxplot ELO_diff por outcome
color_map = {'home_win': '#C8102E', 'draw': '#888888', 'away_win': '#003087'}

fig = px.box(wc_elo, x='outcome', y='elo_diff',
             color='outcome', color_discrete_map=color_map,
             title='Distribución de ELO_diff por resultado (partidos de Mundial)',
             category_orders={'outcome': ['home_win', 'draw', 'away_win']})
fig.add_hline(y=0, line_dash='dash', line_color='white')
fig.update_layout(showlegend=False, yaxis_title='ELO_diff (home - away)')
fig.show()

## 5. Win rate por cuartil de ELO_diff

In [ ]:
wc_elo['elo_q'] = pd.qcut(wc_elo['elo_diff'], q=4,
                           labels=['Q1 (local débil)', 'Q2', 'Q3', 'Q4 (local fuerte)'])

wr = wc_elo.groupby('elo_q', observed=True)['outcome'].value_counts(normalize=True).mul(100).round(1).unstack(fill_value=0)
wr = wr[['home_win', 'draw', 'away_win']]
print(wr.to_string())

fig = go.Figure()
for outcome, color in color_map.items():
    if outcome in wr.columns:
        fig.add_trace(go.Bar(
            name=outcome, x=wr.index.astype(str), y=wr[outcome],
            marker_color=color
        ))
fig.update_layout(
    barmode='stack',
    title='% de resultados por cuartil de ELO_diff',
    xaxis_title='Cuartil ELO_diff', yaxis_title='% partidos',
)
fig.show()

## 6. Evolución ELO de selecciones clave

In [ ]:
teams_of_interest = ['Colombia', 'Argentina', 'Brazil', 'France', 'Spain', 'Germany']

records = []
for team in teams_of_interest:
    mask = (df_elo['home_team'] == team) | (df_elo['away_team'] == team)
    team_df = df_elo[mask].copy()
    team_df['team_elo'] = np.where(team_df['home_team'] == team, team_df['elo_home'], team_df['elo_away'])
    team_df['team'] = team
    records.append(team_df[['date', 'team', 'team_elo']])

df_evolution = pd.concat(records).sort_values('date')

# Resample anual para que la gráfica sea legible
df_annual = (
    df_evolution
    .set_index('date')
    .groupby('team')
    .resample('YE')['team_elo']
    .mean()
    .reset_index()
)

fig = px.line(df_annual, x='date', y='team_elo', color='team',
              title='Evolución del ELO anual — selecciones clave',
              labels={'team_elo': 'ELO', 'date': 'Año'})
fig.add_hline(y=1500, line_dash='dot', line_color='gray', annotation_text='ELO base')
fig.update_layout(hovermode='x unified')
fig.show()

## 7. Colombia — trayectoria ELO histórica

In [ ]:
col_mask = (df_elo['home_team'] == 'Colombia') | (df_elo['away_team'] == 'Colombia')
col_elo_ts = df_elo[col_mask].copy()
col_elo_ts['col_elo'] = np.where(
    col_elo_ts['home_team'] == 'Colombia',
    col_elo_ts['elo_home'], col_elo_ts['elo_away']
)

# ELO de Colombia al inicio de cada Mundial en que participó
wc_col = wc_elo[(wc_elo['home_team'] == 'Colombia') | (wc_elo['away_team'] == 'Colombia')].copy()
wc_col['col_elo'] = np.where(wc_col['home_team'] == 'Colombia', wc_col['elo_home'], wc_col['elo_away'])
col_wc_elo = wc_col.sort_values('date').groupby('year')['col_elo'].first()

print("ELO de Colombia al inicio de cada Mundial:")
print(col_wc_elo.round(1).to_string())
print(f"\nELO actual (junio 2026): {ratings.get('Colombia', 0):.1f}")

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=col_elo_ts['date'], y=col_elo_ts['col_elo'],
    mode='lines', line=dict(color='#FCD116', width=1.5),
    name='ELO Colombia'
))
# Marcar cada Mundial
for year, elo in col_wc_elo.items():
    fig.add_annotation(x=f"{year}-06-01", y=elo, text=str(year),
                       showarrow=True, arrowhead=2, font=dict(color='white'))

fig.add_hline(y=1500, line_dash='dot', line_color='gray')
fig.update_layout(
    title='Trayectoria ELO de Colombia (toda la historia)',
    xaxis_title='Fecha', yaxis_title='ELO',
    plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)'
)
fig.show()

## 8. ELO del Grupo K — Mundial 2026
Colombia, Portugal, Uzbekistán, Congo DR

In [ ]:
grupo_k = ['Colombia', 'Portugal', 'Uzbekistan', 'DR Congo']

grupo_k_elo = [
    {'team': t, 'elo': round(ratings.get(t, 1500), 1)}
    for t in grupo_k
]
df_gk = pd.DataFrame(grupo_k_elo).sort_values('elo', ascending=False)
print("Grupo K — ELO actual:")
print(df_gk.to_string(index=False))

fig = px.bar(df_gk, x='team', y='elo',
             title='ELO actual — Grupo K del Mundial 2026',
             color='elo', color_continuous_scale='RdBu',
             text='elo')
fig.add_hline(y=1500, line_dash='dot', line_color='gray', annotation_text='ELO base')
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False, yaxis_range=[900, 2100])
fig.show()

## Resumen

| Métrica | Valor |
|---|---|
| Correlación elo_diff vs outcome | 0.402 |
| Win rate local Q4 (ELO fuerte) | 68.9% |
| Win rate local Q1 (ELO débil) | 19.5% |
| Colombia ELO actual | 1930.5 (#8 mundial) |
| Colombia ELO en Qatar 2022 | — (no clasificó) |
| Colombia ELO en Rusia 2018 | 1864.3 |

**Conclusión:** el ELO_diff es la feature más importante del modelo. La siguiente sesión construye las features de forma reciente (H2H, goles promedio últimos 5 partidos).